# Hybrid Privacy-Preserving AI Full Walkthrough

This notebook walks through the complete system:
1. Train the loan approval model
2. Run baseline prediction (no privacy)
3. Add Differential Privacy to outputs
4. Encrypt inputs with Homomorphic Encryption
5. Run inference on encrypted data
6. Compare results across all three modes

In [ ]:
import sys
sys.path.insert(0, '../backend')

import numpy as np
import pandas as pd
import tenseal as ts
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✓')

## 1. Train the Model

In [ ]:
from model import train_model, load_model, predict_plain, get_model_weights

model, scaler = train_model()
weights, bias = get_model_weights(model, scaler)

print(f'\nModel weights: {[round(w, 4) for w in weights]}')
print(f'Bias: {round(bias, 4)}')

## 2. Baseline Prediction (No Privacy)

In [ ]:
# Sample applicant
applicant = {
    'income': 50000,
    'credit_score': 720,
    'debt': 10000,
    'employment_years': 5
}

features = list(applicant.values())
plain_prob = predict_plain(model, scaler, features)

print('=== Baseline Prediction ===')
print(f'Inputs (visible to server): {applicant}')
print(f'Approval probability: {plain_prob:.4f}')
print(f'Decision: {"Approved" if plain_prob >= 0.5 else "Rejected"}')

## 3. Add Differential Privacy to Output

In [ ]:
from privacy import DifferentialPrivacy

dp = DifferentialPrivacy(epsilon=1.0, sensitivity=1.0)

# Show effect of noise over multiple runs
n_samples = 100
noisy_probs = [dp.privatize(plain_prob) for _ in range(n_samples)]

print(f'True probability:       {plain_prob:.4f}')
print(f'Mean noisy probability: {np.mean(noisy_probs):.4f}  (should be close to true)')
print(f'Std of noise:           {np.std(noisy_probs):.4f}')
print(f'Min / Max:              {min(noisy_probs):.4f} / {max(noisy_probs):.4f}')

plt.figure(figsize=(8, 3))
plt.hist(noisy_probs, bins=25, color='#1a1a2e', alpha=0.8)
plt.axvline(plain_prob, color='red', linestyle='--', label=f'True = {plain_prob:.3f}')
plt.xlabel('Noisy Probability')
plt.ylabel('Count')
plt.title('Distribution of DP Outputs (ε=1.0, 100 samples)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Homomorphic Encryption

In [ ]:
from encryption import create_context, encrypt_vector, decrypt_vector, sigmoid_approx
import time

# Create HE context
context = create_context()

# Scale features (same as what model expects)
scaled = scaler.transform([features])[0].tolist()
print(f'Scaled features: {[round(v, 4) for v in scaled]}')

# Encrypt
t0 = time.time()
enc_vec = encrypt_vector(context, scaled)
enc_time = time.time() - t0

print(f'\nEncrypted in {enc_time*1000:.1f}ms')
print(f'Ciphertext type: {type(enc_vec)}')
print('Server sees: [encrypted ciphertext — cannot read values]')

In [ ]:
# HE inference: dot product on encrypted data
t0 = time.time()
enc_result = enc_vec.dot(weights)
he_time = time.time() - t0

# Decrypt only the logit
logit = decrypt_vector(enc_result)[0] + bias
he_prob = sigmoid_approx(logit)

print(f'HE dot product time: {he_time*1000:.1f}ms')
print(f'Decrypted logit: {logit:.6f}')
print(f'HE prediction:   {he_prob:.4f}')
print(f'Plain prediction:{plain_prob:.4f}')
print(f'Error: {abs(he_prob - plain_prob):.6f} (should be tiny)')

## 5. Full Pipeline: HE + DP

In [ ]:
# Full pipeline
t_total = time.time()

# Step 1: Encrypt
enc_input = encrypt_vector(context, scaled)

# Step 2: HE inference on server
enc_out = enc_input.dot(weights)
logit = decrypt_vector(enc_out)[0] + bias
raw_prob = sigmoid_approx(logit)

# Step 3: DP noise
final_prob = dp.privatize(raw_prob)

total_time = (time.time() - t_total) * 1000

print('=== Full Privacy Pipeline ===')
print(f'Baseline (no privacy):     {plain_prob:.4f}')
print(f'After HE inference:        {raw_prob:.4f}')
print(f'After DP noise:            {final_prob:.4f}')
print(f'Decision: {"Approved" if final_prob >= 0.5 else "Rejected"}')
print(f'Total latency: {total_time:.1f}ms')

## 6. Privacy Budget: Effect of Epsilon

In [ ]:
epsilons = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
stds = []

for eps in epsilons:
    dp_test = DifferentialPrivacy(epsilon=eps, sensitivity=1.0)
    samples = [dp_test.privatize(plain_prob) for _ in range(500)]
    stds.append(np.std(samples))

plt.figure(figsize=(8, 3))
plt.plot(epsilons, stds, 'o-', color='#1a1a2e', linewidth=2)
plt.xlabel('Epsilon (ε) — Privacy Budget')
plt.ylabel('Output Noise (std)')
plt.title('Trade-off: Privacy vs. Utility\nLower ε = More Private, More Noise')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('ε=0.1 → very high noise (very private)')
print('ε=10  → very low noise (less private)')
print('ε=1.0 → our default (balanced)')